# 01 · Meet CrewAI

> **Kernel:** `Phase 5 · CrewAI` · **Needs:** Ollama.

**CrewAI** builds AI systems the way you'd staff a project: you describe
**agents** as team members (a *role*, a *goal*, a *backstory*), give them
**tasks** with a clear expected output, and put them together in a **crew**
that works through the tasks. For workflows that need explicit control flow,
CrewAI adds **Flows**: event-driven Python classes that decide what runs when.

This notebook gives you the mental model, runs your first crew on a local
model, and shows how CrewAI's philosophy differs from the other frameworks in
this phase.

**You will learn**
- CrewAI's two halves: **Crews** (autonomous teamwork) and **Flows** (explicit control)
- The core vocabulary: Agent, Task, Crew, Process, Flow
- Your first crew on Ollama, and how to read its output
- The one Jupyter rule for this section: `await crew.kickoff_async()`
- When CrewAI is the right choice, and when it isn't

**Prerequisites:** Python and the idea of an LLM calling a tool. *Optional
background:* `00_CONCEPTS/` covers orchestration without any framework.

## 1 · Why CrewAI exists

Other frameworks start from **code primitives**: nodes and edges (LangGraph),
or an agent loop plus Python (OpenAI Agents SDK). CrewAI starts from a
**metaphor**: a team of specialists. You describe *who* does the work and *what*
done looks like, and CrewAI turns that into prompts, agent loops and hand-offs
between tasks.

| You describe… | CrewAI turns it into… |
|---|---|
| an agent's `role`, `goal`, `backstory` | its system prompt: who it is and how it behaves |
| a task's `description` and `expected_output` | the instructions for one unit of work, and the bar for "done" |
| a crew's `process` (sequential or hierarchical) | the order tasks run in, and who delegates |
| task `context` | which earlier results a task receives |

The appeal is **speed and readability**: a multi-agent pipeline in a few dozen
lines that a non-engineer can read. The cost is **less direct control** over
prompts and flow, which is why CrewAI also has **Flows**, where *you* write the
control flow in Python and call crews from it.

```
            CREWS  (autonomy)                            FLOWS  (control)
 ┌──────────────────────────────────┐       ┌────────────────────────────────────┐
 │ Agents with roles + tools        │       │ class MyFlow(Flow[State]):         │
 │ Tasks with expected outputs      │  ◄──  │   @start()  → @listen() → @router()│
 │ Process: sequential/hierarchical │ calls │ explicit state, branching, loops,  │
 │ "the team figures it out"        │       │ persistence, human feedback        │
 └──────────────────────────────────┘       └────────────────────────────────────┘
```

## 2 · Your first crew

Every notebook in this section starts the same way. The setup cell finds this
section's folder and imports `common.py` **before** CrewAI, because `common.py`
switches off CrewAI's anonymous usage telemetry, which is read when CrewAI is
first imported. `get_llm()` returns the local model (notebook 03 explains it).

In [ ]:
import sys, pathlib, warnings
warnings.filterwarnings('ignore')                       # hide noisy library warnings in the notebook
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 03_CREWAI folder
sys.path.insert(0, str(SECTION))
from common import get_llm, quiet_console              # import common BEFORE crewai (telemetry off)
quiet_console()                                         # hide CrewAI's decorative console panels

from crewai import Agent, Task, Crew, Process

llm = get_llm()
print(llm.model)

An **agent** is a persona plus capabilities. A **task** is a unit of work with
a description and an expected output. A **crew** assigns tasks to agents and
runs them:

In [ ]:
explainer = Agent(
    role='Technical explainer',
    goal='Explain technical ideas so that a first-year student understands them',
    backstory='You are a patient teacher who uses one everyday analogy per idea.',
    llm=llm,
)

explain = Task(
    description='Explain what {concept} means in software.',
    expected_output='Two sentences: a definition, then an everyday analogy.',
    agent=explainer,
)

crew = Crew(agents=[explainer], tasks=[explain], process=Process.sequential)
result = await crew.kickoff_async(inputs={'concept': 'an API'})
print(result.raw)

Three things to notice:

1. **`{concept}` in the description** was filled from `inputs`. Tasks and agents
   are templates, so the same crew runs on different inputs.
2. **`await crew.kickoff_async(...)`.** In a plain script you'd call
   `crew.kickoff(...)`. Inside Jupyter an event loop is already running, and in
   this version the synchronous `kickoff()` refuses to run there: it raises an
   error telling you to use `kickoff_async()`. Every notebook in this section uses
   `await crew.kickoff_async()`.
3. **The persona shaped the answer.** The backstory asked for an analogy, and it got one.

## 3 · A two-agent crew: tasks passing results

The real idea is a *team*. Two agents, two tasks: a researcher lists facts, and a
writer turns them into a short note. In a **sequential** crew, each task
automatically receives the previous task's output as context:

In [ ]:
researcher = Agent(role='Researcher', goal='Find the key facts about a topic',
                   backstory='You list facts precisely and never pad them.', llm=llm)
writer = Agent(role='Writer', goal='Turn facts into a clear, friendly note',
               backstory='You write short notes for busy readers.', llm=llm)

facts = Task(description='List 3 key facts about {topic}.',
             expected_output='A bulleted list of exactly 3 facts.', agent=researcher)
note = Task(description='Write a 3-sentence note for a customer newsletter using the facts.',
            expected_output='A 3-sentence note.', agent=writer)

team = Crew(agents=[researcher, writer], tasks=[facts, note], process=Process.sequential)
result = await team.kickoff_async(inputs={'topic': 'sidewalk delivery robots'})

for task_output in result.tasks_output:
    print(f'── {task_output.agent} ──\n{task_output.raw}\n')

`result` is a **`CrewOutput`**: `result.raw` is the last task's text,
`result.tasks_output` holds every task's output (with the agent that produced it),
and `result.token_usage` counts the tokens:

In [ ]:
print('final (raw)  :', result.raw[:80], '…')
print('tasks        :', [t.agent for t in result.tasks_output])
print('token usage  :', result.token_usage.total_tokens, 'tokens over', result.token_usage.successful_requests, 'model calls')

## 4 · How it compares

If you've done other sections, this orients you. Otherwise, skip it.

| | CrewAI | LangGraph | OpenAI Agents SDK |
|---|---|---|---|
| You mainly describe | roles, goals, tasks | a graph of nodes and edges | agents + Python |
| Multi-agent by default | yes: crews are teams | you build it | handoffs / agents-as-tools |
| Explicit control flow | Flows | the graph itself | ordinary Python |
| Prompt control | CrewAI builds prompts from your persona | fully yours | mostly yours |
| Durable state | Flow persistence, checkpoints | checkpoints every step | sessions; the rest is yours |

## 5 · What this section covers

| # | Notebook | You'll be able to… |
|---|---|---|
| 01 | Meet CrewAI | run a crew (this notebook) |
| 02 | Agents and tasks | design personas and tasks; pass inputs and context; read outputs |
| 03 | LLMs and structured output | connect models; get typed results that follow your rubric |
| 04 | Tools | write tools (function and class), handle errors, use a tool's result as the answer |
| 05 | Processes and collaboration | sequential, parallel and hierarchical crews, and what each costs |
| 06 | Flows | explicit control flow with state, routers, joins and loops |
| 07 | Guardrails | validate task outputs and retry, with code or an LLM judge |
| 08 | Human-in-the-loop | pause a flow for a human, save it, resume later |
| 09 | Memory and persistence | remember across runs; what survives a crash |
| 10 | Observability | verbose runs, callbacks, event listeners, token usage |
| 11 | MCP tools | tools from MCP servers, and a pitfall in this version |
| 12 | Production concerns | limits, errors, rate limits, testing with a fake LLM |
| ⭐ | Project | the Nimbus Briefing Assistant as a Flow with crews |

## 6 · When to choose CrewAI

| Choose it when… | Consider something else when… |
|---|---|
| the problem maps naturally to a team of roles | you need exact control of every prompt and step (LangGraph) |
| you want a readable multi-agent pipeline fast | a single agent with tools is enough (Agents SDK) |
| non-engineers should be able to read the design | long-running workflows must resume exactly after any crash (LangGraph, Temporal) |
| you want Flows + crews in one framework | you need minimal dependencies (CrewAI is a large install) |

## 7 · Common mistakes (beginner edition)

- **`crew.kickoff()` in Jupyter.** Use `await crew.kickoff_async()`.
- **Vague `expected_output`.** It's the task's definition of done; "a report" gets you anything.
- **Personas that fight the task.** A "wordy marketer" backstory will fight a
  "max 20 words" requirement (notebook 07 measures it).
- **Leaving telemetry on unknowingly.** CrewAI sends anonymous usage data by default;
  `common.py` turns it off.

## 8 · Check yourself

<details><summary><b>1.</b> What are CrewAI's two halves, and what is each for?</summary>

Crews: teams of role-played agents working through tasks with some autonomy. Flows: event-driven Python classes with explicit state and control flow, which can call crews.
</details>

<details><summary><b>2.</b> Where does an agent's system prompt come from in CrewAI?</summary>

Mostly from its `role`, `goal` and `backstory`, which CrewAI turns into the prompt, together with the task's description and expected output.
</details>

<details><summary><b>3.</b> In a sequential crew, how did the writer get the researcher's facts?</summary>

Sequential crews pass each task's output to the next task as context automatically. (Notebook 02 shows how to choose the context explicitly.)
</details>

<details><summary><b>4.</b> Why does this section use <code>await crew.kickoff_async()</code>?</summary>

Jupyter already runs an event loop, and CrewAI's synchronous `kickoff()` refuses to run inside one. The async version works in notebooks and async apps.
</details>

## Takeaway

CrewAI describes work as a **team**: agents (role, goal, backstory), tasks
(description, expected output) and a crew that runs them in a process. **Flows**
add explicit, stateful control flow around crews. In notebooks, always
`await crew.kickoff_async()`.

Next → `02_agents_and_tasks.ipynb`: designing agents and tasks well.